In [2]:
import pandas as pd
import os
import glob
from datetime import datetime 
from openpyxl import Workbook  
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill  
from openpyxl.utils import get_column_letter
import calendar

# ---- change this to the current month (short name) ----
current_month = 'Aug' 

# set folder path for source files (this folder should only include the files you want to look at)
folder_path = 'L:/2026 Pilar Plant Files/OTS Trends/_OTS Trend Source Files'
# get all xlsx files in the source folder
files = glob.glob(f'{folder_path}/*.xlsx')

# specify sheet names to read  
ap_sheet_name = 'APVCHR'  # change this to the tab name you need
je_sheet_name = 'JRNLLAYOUT'  # change this to the tab name you need

# create empty dataframes
ap_dfs, je_dfs, bud_dfs = [], [], []
# for loop!
for f in files:  
    try:  
        # read the ap files (must have _ap_ in filename, case insensitive)
        if '_ap_' in os.path.basename(f).lower():  
            ap_dfs.append(pd.read_excel(f, sheet_name=ap_sheet_name, skiprows=3))  
        # read the je files (must have _je_ in filename, case insensitive)
        elif '_je_' in os.path.basename(f).lower():  
            je_dfs.append(pd.read_excel(f, sheet_name=je_sheet_name, skiprows=4))
        # read the budget files (must have _bud_ in filename, case insensitive) 
        elif '_bud_' in os.path.basename(f).lower():  
            bud_dfs.append(pd.read_excel(f, sheet_name=je_sheet_name, skiprows=4))  
    # this will skip anything you have open
    except (PermissionError, ValueError):
        pass

# this puts all the dataframes from all the files together into ap, je, and budget dfs
ap_data = pd.concat(ap_dfs, ignore_index=True)  
je_data = pd.concat(je_dfs, ignore_index=True)
bud_data = pd.concat(bud_dfs, ignore_index=True)  

# function to select columns for budget and je
def select_col(df):
    df_new = df[['Line BU', 'Header BU', 'Journal', 'Line', 'Acct', 'Dept', 'Prod', 'Proj', 'Affl', 'Monetary Amt', 'Srce', 'Date', 'Journal Line Description', 
                   'User ID', 'Ledger', 'Year', 'Per', 'Posted Time Stamp', 'Header Long Descr']]
    return df_new

# new dataframes with only necessary columns
ap_data_n = ap_data[ap_data['GL Unit'].notna()].drop(columns=['%,OH'])
je_data_n = select_col(je_data[je_data['Line BU'].notna()])
bud_data_n = select_col(bud_data[bud_data['Line BU'].notna()])

# adding actual and budget indicators
je_data_n['Type'] = 'Actual'
bud_data_n['Type'] = 'Budget'

# combine budget and je dfs for pivot later
journals_all = pd.concat([je_data_n, bud_data_n])

# add month columns from period
journals_all['Month'] = journals_all['Per'].astype(int).apply(lambda x: calendar.month_abbr[x]) 
# update month order so the pivot is in the right order
month_order = list(calendar.month_abbr[1:])  
journals_all['Month'] = pd.Categorical(journals_all['Month'], categories=month_order, ordered=True)

# update type order so budgets come before actuals
type_order = sorted(journals_all['Type'].unique(), reverse=True)
journals_all['Type'] = pd.Categorical(journals_all['Type'], categories=type_order, ordered=True)

In [3]:
# crosswalk file
xwalks = 'L:/2026 Pilar Plant Files/Crosswalks/Manhattan Region Crosswalk.xlsx'

# crosswalk for each column
account_xwalk = pd.read_excel(xwalks, sheet_name='accounts_pl_grouping')
dept_xwalk = pd.read_excel(xwalks, sheet_name='departments')
dir_xwalk = pd.read_excel(xwalks, sheet_name='vp_dept')

# merge each crosswalk with the journals
journals_accounts = pd.merge(account_xwalk, journals_all, how='right', left_on='Account', right_on='Acct')
journals_depts = pd.merge(dept_xwalk, journals_accounts, how='right', on='Dept')
journals_dirs = pd.merge(dir_xwalk, journals_depts, how='right', on='Dept')

# update names for columns
journals_named = journals_dirs.rename(columns={'Dept': 'Dept Number', 'P&L Bucket': 'OTS Grouping', 'Description': 'Account Name'})
# make sure no nulls in amounts
journals_named['Monetary Amt'] = journals_named['Monetary Amt'].fillna(0)
# select needed columns for journals
journals = journals_named[['VP', 'Dept Number', 'Dept Name', 'OTS Grouping', 'Account', 'Account Name', 'Acct', 'Type', 'Per', 'Monetary Amt', 'Line BU', 'Acct']]

# split journals by site
gvccc_journals = journals_named[journals_named['Line BU'] == 'GVCCC']
lenox_journals = journals_named[journals_named['Line BU'] == 'LENOX']
meeth_journals = journals_named[journals_named['Line BU'] == 'MEETH']

# split site journals by type
gvccc_je = gvccc_journals[gvccc_journals['Type'] == 'Actual']
lenox_je = lenox_journals[lenox_journals['Type'] == 'Actual']
meeth_je = meeth_journals[meeth_journals['Type'] == 'Actual']

# merge dept and acct crosswalks with ap
ap_accts = pd.merge(account_xwalk, ap_data_n, how='right', on='Account')
ap_depts = pd.merge(dept_xwalk, ap_accts, how='right', on='Dept')

# rename columns as needed
ap_all = ap_depts.rename(columns={'Description_x': 'Account Name', 'Description_y': 'Description'})

# split ap by site
gvccc_ap = ap_all[ap_all['GL Unit'] == 'GVCCC']
lenox_ap = ap_all[ap_all['GL Unit'] == 'LENOX']
meeth_ap = ap_all[ap_all['GL Unit'] == 'MEETH']

In [4]:
# checks!!

# make sure there are only distinct account and department number combinations
def check_many_to_one(df, site_name):  
    key_cols = ['Dept Number', 'Account']  
    check_cols = ['Account Name', 'OTS Grouping', 'VP', 'Dept Name']  
    
    issues_found = False  
    for col in check_cols:  
        if col not in df.columns:  
            continue  
        mapping = df.groupby(key_cols)[col].nunique().reset_index(name='unique_count')  
        bad = mapping[mapping['unique_count'] > 1]  
          
        if len(bad) > 0:  
            issues_found = True  
            print(f"\n⚠️ {site_name}: {len(bad)} Dept/Account combos map to MULTIPLE '{col}' values:")  
            # Show the conflicting values  
            for _, row in bad.iterrows():  
                subset = df[(df['Dept Number'] == row['Dept Number']) & (df['Account'] == row['Account'])]  
                vals = subset[col].unique()  
                print(f"   Dept {row['Dept Number']} / Account {row['Account']} → {col}: {vals}")  
      
    if not issues_found:  
        print(f"\n✅ {site_name}: All Dept/Account combos map to unique values. No conflicts.")

check_many_to_one(gvccc_journals, 'GVCCC')  
check_many_to_one(lenox_journals, 'LENOX')  
check_many_to_one(meeth_journals, 'MEETH')  

# make sure there are no nulls in any of the pivot columns
def check_nulls(df, site_name):  
    check_cols = ['VP', 'Dept Number', 'Dept Name', 'OTS Grouping', 'Account', 'Account Name']  
    nulls = df[check_cols].isnull().sum()  
    nulls = nulls[nulls > 0]  
      
    if len(nulls) > 0:  
        print(f"\n⚠️ {site_name}: Null values found:")  
        print(nulls)  
        # Show the rows with nulls  
        rows_with_nulls = df[df[check_cols].isnull().any(axis=1)]  
        print(f"\n   {len(rows_with_nulls)} rows affected:")  
        display(rows_with_nulls[check_cols + ['Acct', 'Dept Number', 'Type', 'Monetary Amt']].drop_duplicates())  
    else:  
        print(f"\n✅ {site_name}: No null values in key columns.")

check_nulls(gvccc_journals, 'GVCCC')  
check_nulls(lenox_journals, 'LENOX')  
check_nulls(meeth_journals, 'MEETH')


✅ GVCCC: All Dept/Account combos map to unique values. No conflicts.

✅ LENOX: All Dept/Account combos map to unique values. No conflicts.

✅ MEETH: All Dept/Account combos map to unique values. No conflicts.

✅ GVCCC: No null values in key columns.

✅ LENOX: No null values in key columns.

✅ MEETH: No null values in key columns.


In [5]:
# get current month number from the current month
current_month_num = list(calendar.month_abbr).index(current_month)
# list all the months so far
ytd_months = list(calendar.month_abbr[1:current_month_num + 1])  

# build pivot table
def build_pivot_with_totals(data, current_month=current_month):  
    # get all months
    all_months = list(calendar.month_abbr[1:])  

    # create pivot  
    pivot = pd.pivot_table(  
        data,  
        values='Monetary Amt',  
        index=['VP', 'Dept Number', 'Dept Name', 'OTS Grouping', 'Account', 'Account Name'],  
        columns=['Type', 'Month'],  
        aggfunc='sum' 
    ).reset_index()

    # identify col groups  
    index_cols = [col for col in pivot.columns if col[1] == '']  
    non_index_cols = [col for col in pivot.columns if col[1] != '']  
    # fill non index columns with zeros
    pivot[non_index_cols] = pivot[non_index_cols].fillna(0)

    # sort he budget col by month
    budget_cols = sorted(  
        [col for col in pivot.columns if col[0] == 'Budget'],  
        key=lambda x: all_months.index(x[1])  
    )

    # sort the actual cols by month
    actual_cols = sorted(  
        [col for col in pivot.columns if col[0] == 'Actual' and col[1] in ytd_months],  
        key=lambda x: all_months.index(x[1])  
    )

    # list the ytd budget cols
    ytd_budget_cols = [col for col in budget_cols if col[1] in ytd_months]

    # ---- create calculated columns ----
    # actual total
    pivot[('Total', '')] = pivot[actual_cols].sum(axis=1)

    # mtd actual  
    cur_actual = ('Actual', current_month)  
    pivot[('Actual MTD', '')] = pivot[cur_actual] if cur_actual in pivot.columns else 0

    # mtd budget 
    cur_budget = ('Budget', current_month)  
    pivot[('Budget MTD', '')] = pivot[cur_budget] if cur_budget in pivot.columns else 0

    # mtd variance 
    pivot[('Var MTD', '')] = pivot[('Budget MTD', '')] - pivot[('Actual MTD', '')]

    # ytd actual 
    pivot[('Actual YTD', '')] = pivot[actual_cols].sum(axis=1)

    # ytd budget
    pivot[('Budget YTD', '')] = pivot[ytd_budget_cols].sum(axis=1)

    # ytd variance  
    pivot[('Var YTD', '')] = pivot[('Budget YTD', '')] - pivot[('Actual YTD', '')]

    # reorder columns to match needed output
    ordered_columns = (  
        index_cols + 
        actual_cols +  
        [('Total', '')] +  
        [('Actual MTD', '')] +  
        [('Budget MTD', '')] +  
        [('Var MTD', '')] +  
        [('Actual YTD', '')] +  
        [('Budget YTD', '')] +  
        [('Var YTD', '')]  
    )

    ordered_columns = [col for col in ordered_columns if col in pivot.columns]  
    pivot = pivot[ordered_columns]

    # flatten columns names for df  
    clean_names = []  
    for col in pivot.columns:  
        if col[1] == '':  
            clean_names.append(col[0])  
        else:  
            clean_names.append(col[1])  # Just month name  
    pivot.columns = clean_names

    # add comments columns
    pivot['Comments'] = ''

    return pivot

# pivot journals
pivot_lenox = build_pivot_with_totals(lenox_journals).fillna(0) 
pivot_gvccc = build_pivot_with_totals(gvccc_journals).fillna(0)   
pivot_meeth = build_pivot_with_totals(meeth_journals).fillna(0)

In [ ]:
# define hospital names
hospital_names = {  
    'LENOX': 'Lenox Hill Hospital',  
    'GVCCC': 'Greenwich Village Hospital',  
    'MEETH': 'Manhattan Eye, Ear and Throat Hospital'  
}

# define number format for $
number_format = '_($* #,##0_);_($* (#,##0);_($* "-"??_);_(@_)'

def format_data_sheet(ws):  
    thin_bottom_border = Border(bottom=Side(style='thin'))

    # Format header row  
    for cell in ws[1]:  
        cell.font = Font(name='Ebrima', size=10, bold=True)  
        cell.border = thin_bottom_border  
    ws.freeze_panes = 'A2'

    # Format data rows (general font)  
    for row in ws.iter_rows(min_row=2):  
        for cell in row:  
            cell.font = Font(name='Ebrima', size=10)

    # --- Add Grand Total and Column Formatting ---

    # 1. Identify the "monetary amt" column  
    monetary_col_idx = None  
    monetary_col_letter = None  
    # Add/remove headers as needed to match your sheet  
    monetary_headers = ['Monetary Amount', 'Monetary Amt']

    for col_idx, cell in enumerate(ws[1], 1): # Iterate through header row (row 1)  
        if cell.value and str(cell.value).strip().lower() in [h.lower() for h in monetary_headers]:  
            monetary_col_idx = col_idx  
            monetary_col_letter = get_column_letter(col_idx)  
            break

    if monetary_col_idx is None:  
        print(f"Warning: Could not find a 'monetary amt' column with headers: {monetary_headers}. Grand Total not added.")  
        return # Exit if the column isn't found

    # 2. Determine the last row with data  
    last_data_row = ws.max_row  
    total_row_num = last_data_row + 1

    # Define the specific border for the grand total row  
    total_row_style_border = Border(  
        top=Side(style='thin'),  
        bottom=Side(style='double')  
    )

    
    # --- Apply currency formatting to the entire "Monetary Amt" column ---  
    # Iterate from the first data row (2) up to the last data row,  
    # and then also apply it to the grand total cell.  
    for row_idx in range(2, total_row_num + 1):  
        cell_to_format = ws.cell(row=row_idx, column=monetary_col_idx)  
        cell_to_format.number_format = number_format


    # 3. Insert "Grand Total" label  
    label_cell = ws.cell(row=total_row_num, column=1) # Explicitly column 1 (A)  
    label_cell.value = "Grand Total"  
    label_cell.font = Font(name='Ebrima', size=10, bold=True)  
    # Apply the border to the label cell  
    label_cell.border = total_row_style_border


    # 4. Insert the SUM formula  
    sum_range = f'{monetary_col_letter}2:{monetary_col_letter}{last_data_row}'  
    total_cell = ws[f'{monetary_col_letter}{total_row_num}']  
    total_cell.value = f'=SUBTOTAL(9,{sum_range})'

    # 5. Apply formatting to the total cell  
    total_cell.font = Font(name='Ebrima', size=10, bold=True)  
    # The number_format is already applied in the loop above for consistency,  
    # but we'll make sure it's set here too in case the loop range changes.  
    total_cell.number_format = number_format  
    # Apply the border to the total cell  
    total_cell.border = total_row_style_border


    last_col_with_data = ws.max_column

    for col_idx in range(1, last_col_with_data + 1):  
        cell = ws.cell(row=total_row_num, column=col_idx)  
        cell.border = total_row_style_border
    

# format the trend sheet
def format_trend_sheet(ws, df, site_key, current_month):  
    hospital_name = hospital_names.get(site_key, site_key)

    index_cols = ['VP', 'Dept Number', 'Dept Name', 'OTS Grouping', 'Account', 'Account Name']  
    all_cols = list(df.columns)  
    num_cols = [c for c in all_cols if c not in index_cols]

    mtd_cols = ['Actual MTD', 'Budget MTD', 'Var MTD']  
    ytd_cols = ['Actual YTD', 'Budget YTD', 'Var YTD']  
    mtd_indices = [all_cols.index(c) + 1 for c in mtd_cols if c in all_cols]  
    ytd_indices = [all_cols.index(c) + 1 for c in ytd_cols if c in all_cols]

    ws.delete_rows(1, ws.max_row)

    # --- Row 1: Hospital Name (bold) ---  
    ws.cell(row=1, column=1, value=hospital_name).font = Font(name='Ebrima', size=10, bold=True)

    # --- Row 2: "OTS Trend" (italic) ---  
    ws.cell(row=2, column=1, value='OTS Trend').font = Font(name='Ebrima', size=10, italic=True)

    # --- Row 3: "{Month} YTD 2026" (italic) ---  
    ws.cell(row=3, column=1, value=f'{current_month} YTD 2026').font = Font(name='Ebrima', size=10, italic=True)

    # --- Row 5: MTD and YTD group headers ---  
    medium_border_side = Side(style='medium')  
    header_font = Font(name='Ebrima', size=10, bold=True)  
    grey_fill = PatternFill(start_color='D9D9D9', end_color='D9D9D9', fill_type='solid')

    if mtd_indices:  
        mtd_start = min(mtd_indices)  
        mtd_end = max(mtd_indices)  
        ws.cell(row=5, column=mtd_start, value=f'{current_month} MTD').font = header_font

        for col_idx in range(mtd_start, mtd_end + 1):  
            cell = ws.cell(row=5, column=col_idx)  
            cell.alignment = Alignment(horizontal='centerContinuous')  
            cell.font = header_font  
            left = medium_border_side if col_idx == mtd_start else Side()  
            right = medium_border_side if col_idx == mtd_end else Side()  
            cell.border = Border(  
                top=medium_border_side,  
                bottom=medium_border_side,  
                left=left,  
                right=right  
            )

    if ytd_indices:  
        ytd_start = min(ytd_indices)  
        ytd_end = max(ytd_indices)  
        ws.cell(row=5, column=ytd_start, value=f'{current_month} YTD').font = header_font

        for col_idx in range(ytd_start, ytd_end + 1):  
            cell = ws.cell(row=5, column=col_idx)  
            cell.alignment = Alignment(horizontal='centerContinuous')  
            cell.font = header_font  
            left = medium_border_side if col_idx == ytd_start else Side()  
            right = medium_border_side if col_idx == ytd_end else Side()  
            cell.border = Border(  
                top=medium_border_side,  
                bottom=medium_border_side,  
                left=left,  
                right=right  
            )

    # --- Row 6: Column headers (bold, grey fill, centered) ---  
    header_row = 6  
    for col_idx, col_name in enumerate(all_cols, 1):  
        cell = ws.cell(row=header_row, column=col_idx, value=col_name)  
        cell.font = Font(name='Ebrima', size=10, bold=True)  
        cell.fill = grey_fill  
        cell.alignment = Alignment(horizontal='center')

    # ============================================================  
    # WRITE DATA WITH SUBTOTALS AND GROUPING  
    # ============================================================  
    data_start_row = header_row + 1  
    current_row = data_start_row

    thin_top_border = Border(top=Side(style='thin'))  
    grand_total_border = Border(top=Side(style='thin'), bottom=Side(style='double'))  
    subtotal_font = Font(name='Ebrima', size=10, bold=True)  
    default_font = Font(name='Ebrima', size=10)

    df_sorted = df.sort_values(['VP', 'Dept Number']).reset_index(drop=True)

    # Track numeric column letters for formulas  
    num_col_letters = {}  
    for col_name in num_cols:  
        col_idx = all_cols.index(col_name) + 1  
        num_col_letters[col_name] = get_column_letter(col_idx)

    grouped_vp = df_sorted.groupby('VP', sort=False)

    vp_subtotal_rows = []  # Track VP subtotal rows for grand total formula

    for vp_name, vp_group in grouped_vp:  
        vp_start_row = current_row  
        dept_subtotal_rows = []  
        grouped_dept = vp_group.groupby(['Dept Number', 'Dept Name'], sort=False)

        for (dept_num, dept_name), dept_group in grouped_dept:  
            dept_start_row = current_row

            # Write data rows  
            for _, row in dept_group.iterrows():  
                for col_idx, col_name in enumerate(all_cols, 1):  
                    cell = ws.cell(row=current_row, column=col_idx, value=row[col_name])  
                    cell.font = default_font  
                    if col_name in num_cols:  
                        cell.number_format = number_format  
                current_row += 1

            dept_end_row = current_row - 1

            # --- Dept subtotal with SUBTOTAL formulas ---  
            dept_subtotal_row = current_row  
            dept_subtotal_rows.append(dept_subtotal_row)

            for col_idx, col_name in enumerate(all_cols, 1):  
                cell = ws.cell(row=current_row, column=col_idx)  
                cell.font = subtotal_font  
                cell.border = thin_top_border  
                if col_name in num_cols:  
                    col_letter = num_col_letters[col_name]  
                    cell.value = f'=SUBTOTAL(9,{col_letter}{dept_start_row}:{col_letter}{dept_end_row})'  
                    cell.number_format = number_format  
                elif col_name == 'Dept Number':  
                    cell.value = f'{dept_num} - {dept_name}'  
            current_row += 1

            # Group detail rows at level 3  
            if dept_subtotal_row > dept_start_row:  
                ws.row_dimensions.group(dept_start_row, dept_subtotal_row - 1, outline_level=3, hidden=False)

         # --- VP subtotal ---  
        vp_subtotal_row = current_row  
        vp_subtotal_rows.append(vp_subtotal_row)

        for col_idx, col_name in enumerate(all_cols, 1):  
            cell = ws.cell(row=current_row, column=col_idx)  
            cell.font = subtotal_font  
            cell.border = thin_top_border  
            if col_name in num_cols:  
                col_letter = num_col_letters[col_name]  
                # ✅ Use full range from vp_start_row to row above subtotal  
                cell.value = f'=SUBTOTAL(9,{col_letter}{vp_start_row}:{col_letter}{vp_subtotal_row - 1})'  
                cell.number_format = number_format  
            elif col_name == 'VP':  
                cell.value = f'Total {vp_name}'  
        current_row += 1

        # Group dept subtotals + detail at level 2 (under VP)  
        if vp_subtotal_row > vp_start_row:  
            ws.row_dimensions.group(vp_start_row, vp_subtotal_row - 1, outline_level=1, hidden=False)

        
    # We need to re-apply grouping since openpyxl only keeps the last group call per row  
    # Reset all row groups first  
    for row_idx in range(data_start_row, current_row):  
        ws.row_dimensions[row_idx].outlineLevel = 0

    # Now walk through and assign levels  
    df_sorted_reset = df_sorted.reset_index(drop=True)  
    walk_row = data_start_row

    for vp_name, vp_group in df_sorted.groupby('VP', sort=False):  
        vp_block_start = walk_row  
        for (dept_num, dept_name), dept_group in vp_group.groupby(['Dept Number', 'Dept Name'], sort=False):  
            dept_block_start = walk_row  
            num_detail = len(dept_group)

            # Detail rows: level 3  
            for i in range(num_detail):  
                ws.row_dimensions[walk_row].outlineLevel = 3  
                walk_row += 1

            # Dept subtotal row: level 2  
            ws.row_dimensions[walk_row].outlineLevel = 2  
            walk_row += 1

        # VP subtotal row: level 1  
        ws.row_dimensions[walk_row].outlineLevel = 1  
        walk_row += 1

    # ============================================================  
    # GRAND TOTAL ROW  
    # ============================================================  
    grand_total_row = current_row

    for col_idx, col_name in enumerate(all_cols, 1):  
        cell = ws.cell(row=grand_total_row, column=col_idx)  
        cell.font = subtotal_font  
        cell.border = grand_total_border  
        if col_name in num_cols:  
            col_letter = num_col_letters[col_name]  
            # ✅ Use full range from first data row to row above grand total  
            cell.value = f'=SUBTOTAL(9,{col_letter}{data_start_row}:{col_letter}{grand_total_row - 1})'  
            cell.number_format = number_format  
        elif col_name == 'VP':  
            cell.value = 'Grand Total' 

    # Grand total row stays at level 0 (always visible)

    # ============================================================  
    # COLUMN WIDTHS  
    # ============================================================  
    for col_idx, col_name in enumerate(all_cols, 1):  
        col_letter = get_column_letter(col_idx)  
        if col_name == 'Dept Number':  
            ws.column_dimensions[col_letter].width = 13  
        elif col_name in index_cols:  
            max_len = len(str(col_name))  
            for row in ws.iter_rows(min_row=header_row, max_row=ws.max_row, min_col=col_idx, max_col=col_idx):  
                for cell in row:  
                    if cell.value:  
                        max_len = max(max_len, len(str(cell.value)))  
            ws.column_dimensions[col_letter].width = max_len + 2  
        elif col_name == 'Comments':  
            ws.column_dimensions[col_letter].width = 28  
        else:  
            ws.column_dimensions[col_letter].width = 13

    # Freeze panes below header row  
    first_num_col_idx = all_cols.index(num_cols[0]) + 1  
    first_num_col_letter = get_column_letter(first_num_col_idx)  
    ws.freeze_panes = f'{first_num_col_letter}{header_row + 1}'

    # Set outline summary rows below detail  
    ws.sheet_properties.outlinePr.summaryBelow = True  

# ============================================================  
# EXPORT  
# ============================================================  
output_folder = r'L:/2026 Pilar Plant Files/OTS Trends'

datasets = {  
    f'LENOX - {current_month} OTS Trend.xlsx': {'site': 'LENOX', 'Trend': pivot_lenox, 'AP': lenox_ap, 'JE': lenox_je},  
    f'NGVH - {current_month} OTS Trend.xlsx': {'site': 'NGVH', 'Trend': pivot_gvccc, 'AP': gvccc_ap, 'JE': gvccc_je},  
    f'MEETH - {current_month} OTS Trend.xlsx': {'site': 'MEETH', 'Trend': pivot_meeth, 'AP': meeth_ap, 'JE': meeth_je},  
}

for filename, content in datasets.items():  
    site_key = content['site']  
    site_folder = os.path.join(output_folder, site_key)  
    os.makedirs(site_folder, exist_ok=True)  
    filepath = os.path.join(site_folder, filename)

    with pd.ExcelWriter(filepath, engine='openpyxl') as writer:  
        content['Trend'].to_excel(writer, sheet_name='Trend', index=False)  
        content['AP'].to_excel(writer, sheet_name='AP', index=False)
        content['JE'].to_excel(writer, sheet_name='JE', index=False)

        ws_trend = writer.sheets['Trend']  
        format_trend_sheet(ws_trend, content['Trend'], site_key, current_month)

        ws_ap = writer.sheets['AP']  
        format_data_sheet(ws_ap)

        ws_je = writer.sheets['JE']  
        format_data_sheet(ws_je)

    print(f"✅ File saved successfully at: {filepath}") 
    